Grupo 4 - 18/05/2026 - IA - Trabajo IA - Detección y Recorte de Caras

# Notebook 2: Detección y Recorte de Caras

En el Notebook 1 analizamos el dataset de fotos del Gobierno de La Rioja y comprobamos que el 82,2% de las imágenes contienen al menos una cara detectada con Haar Cascades. Sin embargo, como mencionamos, Haar Cascades es un detector rápido pero poco preciso que genera falsos positivos y pierde caras de perfil o con oclusiones,

En este notebook haremos la detección de verdad primero comparando tres detectores. Una vez elegido el mejor recortaremos y normalizaremos las caras de las imagenes del dataset.

## 0. Carga de librerías

Cargamos las librerías necesarias para la deteccion.

In [ ]:
%%capture
!pip install cmake
!pip install dlib
!pip install opencv-contrib-python

In [ ]:
import os
import cv2
import dlib
import numpy as np
import matplotlib.pyplot as plt
import random
import time
import urllib.request
from pathlib import Path
from collections import defaultdict
%matplotlib inline

## 1. Carga del Dataset

Descargamos el dataset desde el mismo release de GitHub que usamos en el Notebook 1.

In [ ]:
%%capture
!wget DATASET_URL -O dados.zip
!unzip dados.zip

In [4]:
datasetPath = Path('/content/2')

imagenes = [p for p in datasetPath.rglob('*') ]

print(f"Total de imágenes: {len(imagenes)}")
print(f"Carpetas de primer nivel: {[f.name for f in sorted(datasetPath.iterdir()) if f.is_dir()][:10]}")

Total de imágenes: 21678
Carpetas de primer nivel: []


## 2. Comparativa de Detectores Faciales

Para evitar procesar el dataset completo con los tres detectores, vamos a comparar los tres detectores sobre una muestra aleatoria de 200 imágenes del dataset para evaluar su precisión visual y velocidad.

Los tres detectores que vamos a comparar son:

| Detector | Tipo | Velocidad | Precisión esperada |
|---|---|---|---|
| Haar Cascades | ML | Muy rápido | Baja  |
| HOG + SVM |  ML | Rápido | Media  |
| CNN (MMOD) |  Deep Learning | Lento  | Alta  |


### 2.1 Detector 1: Haar Cascades

In [5]:
# load the haar cascade face detector
print("[INFO] loading face detector...")
detectorHaar = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def detectarHaar(imagen_bgr):
    gray = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2GRAY)
    rects = detectorHaar.detectMultiScale(gray, scaleFactor=1.05,
        minNeighbors=5, minSize=(30, 30),
        flags=cv2.CASCADE_SCALE_IMAGE)
    return [(x, y, w, h) for (x, y, w, h) in rects] if len(rects) else []

[INFO] loading face detector...


### 2.2 Detector 2: dlib HOG + SVM

In [10]:
# auxiliary function from PyImageSearch helpers.py
def convert_and_trim_bb(image, rect):
    # extract the starting and ending (x, y)-coordinates of the bounding box
    startX = rect.left()
    startY = rect.top()
    endX = rect.right()
    endY = rect.bottom()

    # ensure the bounding box coordinates fall within the spatial
    # dimensions of the image
    startX = max(0, startX)
    startY = max(0, startY)
    endX = min(endX, image.shape[1])
    endY = min(endY, image.shape[0])

    # compute the width and height of the bounding box
    w = endX - startX
    h = endY - startY

    # return our bounding box coordinates
    return (startX, startY, w, h)

# load dlib's HOG + Linear SVM face detector
print("[INFO] loading HOG + Linear SVM face detector...")
detectorHOG = dlib.get_frontal_face_detector()

def detectarHOG(imagen_bgr):
    rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    rects = detectorHOG(rgb, 1)
    return [convert_and_trim_bb(imagen_bgr, r) for r in rects]

[INFO] loading HOG + Linear SVM face detector...


### 2.3 Detector 3: dlib CNN (MMOD)

Descargamos el modelo preentrenado `mmod_human_face_detector.dat`.

In [7]:
!wget https://github.com/davisking/dlib-models/raw/master/mmod_human_face_detector.dat.bz2
!bunzip2 mmod_human_face_detector.dat.bz2

--2026-05-18 14:09:51--  https://github.com/davisking/dlib-models/raw/master/mmod_human_face_detector.dat.bz2
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/davisking/dlib-models/master/mmod_human_face_detector.dat.bz2 [following]
--2026-05-18 14:09:51--  https://raw.githubusercontent.com/davisking/dlib-models/master/mmod_human_face_detector.dat.bz2
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 694687 (678K) [application/octet-stream]
Saving to: ‘mmod_human_face_detector.dat.bz2’

mmod_human_face_det 100%[===================>] 678.41K  --.-KB/s    in 0.003s  

2026-05-18 14:09:52 (199 MB/

In [8]:
print("[INFO] loading CNN face detector...")

cnn_model_path = Path('mmod_human_face_detector.dat')

detectorCNN = dlib.cnn_face_detection_model_v1(str(cnn_model_path))

def detectarCNN(imagen_bgr):
    rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    results = detectorCNN(rgb, 1)
    return [convert_and_trim_bb(imagen_bgr, r.rect) for r in results]

[INFO] loading CNN face detector...


### 2.4 Visualización

Aplicamos los tres detectores sobre una muestra aleatoria de 200 imágenes y pintamos las bounding boxes de 3 imagenes de esta muestra.

In [9]:
random.seed(42)
muestra = random.sample(imagenes,200)

In [10]:
detectores = {'Haar': detectarHaar, 'HOG': detectarHOG, 'CNN': detectarCNN}
colores = [(255,0,0), (0,200,0), (0,0,255)]

In [11]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for fila, ruta in enumerate(muestra[:3]):
    img = cv2.imread(str(ruta))
    for col, (nombre, fn) in enumerate(detectores.items()):
        copia = img.copy()
        caras = fn(copia)
        for (x, y, w, h) in caras:
            cv2.rectangle(copia, (x, y), (x+w, y+h), colores[col], 3)
        axes[fila][col].imshow(cv2.cvtColor(copia, cv2.COLOR_BGR2RGB))
        axes[fila][col].set_title(f"{nombre} — {len(caras)} cara(s)")
        axes[fila][col].axis('off')

plt.suptitle('Comparativa de detectores faciales')
plt.tight_layout()
plt.show()

<Figure size 1500x1200 with 9 Axes>

### 2.5 Evaluacion de rendimiento

Medimos el tiempo de procesado de cada detector sobre las 200 imágenes para tener datos del rendimiento.

In [12]:
imagenes_cargadas = [img for r in muestra[:200] if (img := cv2.imread(str(r))) is not None]
print(f"Imágenes cargadas: {len(imagenes_cargadas)}\n")

for nombre, fn in detectores.items():
    start = time.time()
    caras = sum(len(fn(img)) for img in imagenes_cargadas)
    elapsed = time.time() - start
    print(f"{nombre:<10} {elapsed:.2f}s — {caras} caras")

Imágenes cargadas: 200

Haar       72.22s — 579 caras
HOG        50.49s — 349 caras
CNN        19.42s — 347 caras


### 2.6 Elección del detector final: dlib CNN

| Detector | Tiempo | Caras detectadas |
|---|---|---|
| Haar Cascades | 72.38s | 579 |
| HOG  | 48.31s | 349 |
| CNN  | 19.72s | 347 |

La comparativa visual sobre las 3 imagenes de muestra complementa los números obtenidos, y nos lleva a las siguientes conclusiones:

- Haar tiene el mayor recall, vemos que en la foto de la piscina detecta 4 caras de niños con bounding boxes bastante acertadas, y en la foto de la reunión las 2 cajas principales están bien centradas. Sin embargo incluye falsos positivos claros en ambas fotos, lo que haria que tengamos recortes inútiles para el Notebook 3.
- HOG queda por detrás de los otros dos ya detecta menos caras que Haar sin la ventaja de precisión de CNN.
- CNN no detecta las caras pequeñas de la piscina, pero donde detecta sus bounding boxes son precisas y sin falsos positivos. Además, con GPU de colab es el más rápido de los tres con diferencia.

Elegimos CNN ya que una cara no detectada contamina menos los embeddings del Notebook 3 que un falso positivo con bounding box imprecisa, que le pasaría al modelo de reconocimiento facial un recorte que no representa una cara.Es por estas razones que lo elegimos como detector principal.

## 3. Normalización de Caras

Como vimos en el notebook 1 es necesario normalizarlas a un tamaño fijo antes de guardarlas. Los modelos de reconocimiento facial esperan imágenes de tamaño fijo, por lo que elegimos 160×160 px como resolución de entrada.

En el proceso de normalización por cara lo que haremos es primeramente añadir un margen de 0.2 para incluir toda la cara bien, recortamos la region donde esta esa cara y la redimencionamos a 160x160 px.

In [11]:
def recortarYNormalizar(imagen_bgr, x, y, w, h):
    ih, iw = imagen_bgr.shape[:2]
    margen_px_w = int(w * 0.20)
    margen_px_h = int(h * 0.20)
    x1 = max(0, x - margen_px_w)
    y1 = max(0, y - margen_px_h)
    x2 = min(iw, x + w + margen_px_w)
    y2 = min(ih, y + h + margen_px_h)
    if (x2 - x1) < 20 or (y2 - y1) < 20:
        return None
    recorte = imagen_bgr[y1:y2, x1:x2]
    recorte_norm = cv2.resize(recorte, (160, 160), interpolation=cv2.INTER_CUBIC)
    return recorte_norm

Verificamos visualmente el resultado de la normalización sobre las 10 primeras imágenes de la muestra en lugar de 3, ya que CNN es conservador y con solo 3 imágenes únicamente obteníamos 2 recortes, lo que resultaba poco para visualizar el resultado de la normalización.

In [14]:
todos_recortes = []

for ruta in muestra[:10]:
    img = cv2.imread(str(ruta))
    if img is None:
        continue
    for (x, y, w, h) in detectarCNN(img):
        recorte = recortarYNormalizar(img, x, y, w, h)
        if recorte is not None:
            todos_recortes.append(recorte)

print(f"Recortes obtenidos de la muestra: {len(todos_recortes)}")

Recortes obtenidos de la muestra: 18


In [15]:
n_mostrar = min(16, len(todos_recortes))
fig, axes = plt.subplots(1, n_mostrar, figsize=(3 * n_mostrar, 3))
for i, ax in enumerate(axes):
    ax.imshow(cv2.cvtColor(todos_recortes[i], cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.show()

<Figure size 4800x300 with 16 Axes>

## 4. Procesado del Dataset Completo

Aplicamos el detector CNN sobre las 21.678 imágenes del dataset. Primeramente vamos a intentar hacerlo de forma secuencial.

### 4.1 Procesado del dataset secuencialmente en Google Colab

In [18]:
from collections import defaultdict
contadores = defaultdict(int)

In [16]:
DRIVE_OUT = Path('carasRecortadas')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print(f"Directorio de salida: {DRIVE_OUT}")

Directorio de salida: /content/caras_recortadas


In [ ]:
print(f"Procesando {len(imagenes)} imágenes...")
t_inicio = time.time()

for idx, ruta in enumerate(imagenes):
    img = cv2.imread(str(ruta))
    if img is None:
        contadores['error'] += 1
        continue

    if len(img.shape) == 2: img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4: img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

    try:
        caras = [convert_and_trim_bb(img, r.rect) for r in detectorCNN(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), 1)]
    except Exception:
        contadores['error'] += 1
        continue

    contadores['procesadas'] += 1
    if not caras:
        contadores['sin_cara'] += 1
        continue

    contadores['con_cara'] += 1
    for i, (x, y, w, h) in enumerate(caras):
        recorte = recortarYNormalizar(img, x, y, w, h)
        if recorte is not None:
            cv2.imwrite(str(DRIVE_OUT / f'{ruta.stem}_{i}.jpg'), recorte, [cv2.IMWRITE_JPEG_QUALITY, 95])
            contadores['recortes'] += 1

    if idx % 1000 == 0:
        print(f"  {idx}/{len(imagenes)} — {contadores['recortes']} recortes...")

tf = time.time() - t_inicio

Procesando 21678 imágenes...
  2000/21678 — 3679 recortes guardados...
  3000/21678 — 5448 recortes guardados...
  4000/21678 — 7349 recortes guardados...
  5000/21678 — 9225 recortes guardados...
  8000/21678 — 14930 recortes guardados...
  9000/21678 — 16943 recortes guardados...
  11000/21678 — 20672 recortes guardados...
  12000/21678 — 22508 recortes guardados...


Falla google colab porque se superó la memoria RAM de la sesion de por lo que se reinicia tras ~1 hora de ejecución sin completar el procesado. Vamos a hacerlo loclamente

### 4.2 Procesado del dataset secuencialmente en local con GPU

In [6]:
import cv2
import dlib
import time
from pathlib import Path
from collections import defaultdict

In [7]:
datasetPath = Path('noticiasFotos/2')
imagenes = [p for p in datasetPath.rglob('*') if p.is_file()]
print(f"Imágenes encontradas: {len(imagenes)}")

cnn_model_path = Path('mmod_human_face_detector.dat')
assert cnn_model_path.exists(), "No se encuentra el modelo CNN"
print("Modelo CNN encontrado.")

Imágenes encontradas: 21678
Modelo CNN encontrado.


In [8]:
detector = dlib.cnn_face_detection_model_v1(str(cnn_model_path))
print("Modelo cargado.")

Modelo cargado.


In [9]:
DRIVE_OUT = Path('carasRecortadas')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print(f"Directorio de salida: {DRIVE_OUT}")

Directorio de salida: carasRecortadas


In [12]:
contadores = defaultdict(int)

print(f"Procesando {len(imagenes)} imágenes...")
t_inicio = time.time()

for idx, ruta in enumerate(imagenes):
    img = cv2.imread(str(ruta))
    if img is None:
        contadores['error'] += 1
        continue

    if len(img.shape) == 2: img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4: img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

    try:
        caras = [convert_and_trim_bb(img, r.rect)
                 for r in detector(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), 1)]
    except Exception:
        contadores['error'] += 1
        continue

    contadores['procesadas'] += 1
    if not caras:
        contadores['sin_cara'] += 1
        continue

    contadores['con_cara'] += 1
    for i, (x, y, w, h) in enumerate(caras):
        recorte = recortarYNormalizar(img, x, y, w, h)
        if recorte is not None:
            cv2.imwrite(str(DRIVE_OUT / f'{ruta.stem}_{i}.jpg'), recorte,
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            contadores['recortes'] += 1

    if idx % 500 == 0:
        print(f"  {idx}/{len(imagenes)} — {contadores['recortes']} recortes...")

tf = time.time() - t_inicio

Procesando 21678 imágenes...
  0/21678 — 3 recortes...
  500/21678 — 972 recortes...
  3000/21678 — 5839 recortes...
  3500/21678 — 6889 recortes...
  4000/21678 — 7926 recortes...
  5000/21678 — 9864 recortes...
  5500/21678 — 10852 recortes...
  6500/21678 — 12856 recortes...
  7000/21678 — 13852 recortes...
  7500/21678 — 14800 recortes...
  8000/21678 — 15721 recortes...
  9500/21678 — 18691 recortes...
  11000/21678 — 21624 recortes...
  12000/21678 — 23591 recortes...
  13000/21678 — 25615 recortes...
  13500/21678 — 26593 recortes...
  14500/21678 — 28439 recortes...
  15000/21678 — 29421 recortes...
  15500/21678 — 30321 recortes...
  16000/21678 — 31233 recortes...
  16500/21678 — 32274 recortes...
  17000/21678 — 33271 recortes...
  17500/21678 — 34214 recortes...
  18500/21678 — 36088 recortes...
  19500/21678 — 37461 recortes...
  20000/21678 — 38175 recortes...


In [13]:
print(f"\nCompletado en {tf:.1f}s")
print(f"  Imágenes procesadas : {contadores['procesadas']}")
print(f"  Con cara            : {contadores['con_cara']}")
print(f"  Sin cara            : {contadores['sin_cara']}")
print(f"  Errores             : {contadores['error']}")
print(f"  Recortes guardados  : {contadores['recortes']}")


Completado en 2913.9s
  Imágenes procesadas : 20088
  Con cara            : 13608
  Sin cara            : 6480
  Errores             : 1590
  Recortes guardados  : 38299


#### 4.3 Comentarios sobre el procesado del dataset completo

El procesado de las 21.678 imágenes con dlib CNN resultó ser el paso más
problemático del notebook con mucha diferencia. Aquí documentamos el proceso hasta llegar a la
solución final.

**1. Google Colab (intento inicial)**
La ejecución secuencial en Colab funcionó parcialmente, pero el dataset
supera el límite gratuito de GPU de Colab (~12GB de RAM de GPU), por lo que
la sesión se cortaba antes de terminar.

**2. Ejecución local secuencial en CPU**
dlib CNN sin GPU tarda ~12.7s por imagen en CPU. Con 21.678 imágenes,
el tiempo estimado era de ~70 horas que es inviable.

**3. Paralelismo con ThreadPoolExecutor**
El primer intento de paralelización usó ThreadPoolExecutor. Falló porque
dlib CNN no es thread-safe, el modelo comparte estado interno entre hilos,
lo que provoca resultados corruptos o cuelgues del kernel.

**4. Paralelismo con ProcessPoolExecutor**
La solución al punto 3 era ProcessPoolExecutor, donde cada proceso
tiene su propia copia del modelo en memoria. Sin embargo, en Windows con
Python del Microsoft Store, multiprocessing usa spawn para crear procesos
nuevos, lo que requiere que la función worker esté en un fichero .py
independiente . Aun así, con 15 workers intentando cargar
el modelo CNN en CPU simultáneamente, se agotaba la memoria virtual del sistema.

**5. Solución final: ejecución local secuencial con GPU**
La solución fue instalar localmente CUDA 12.4, cuDNN 9, y Visual Studio
Build Tools para compilar dlib desde código fuente con soporte CUDA activado.
Con GPU, dlib CNN tarda ~0.14s por imagen en lugar de ~12.7s en CPU.
El procesado secuencial con GPU completó las 21.678 imágenes en 48 minutos,
guardando 38.299 recortes de caras.

## 5. Conclusiones

En este notebook hemos completado la detección y recorte de caras sobre el dataset del Gobierno
de La Rioja. Los puntos clave son:

- Comparamos tres detectores (Haar, HOG y CNN) concluyendo que CNN ofrece la mejor precisión,
evitando los falsos positivos de Haar y superando a HOG donde detecta.
- Cada cara detectada se ha recortado con un margen del 20% y normalizada a 160×160 px,
el tamaño estándar de entrada para los modelos de reconocimiento facial que usaremos en el
Notebook 3.
- El nombre de cada fichero conserva el nombre de la imagen original más el índice de la cara
(`{imagen_fuente}_{índice}.jpg`), lo que permite trazar cada recorte hasta su origen.

En el Notebook 3 cargaremos los recortes de `carasRecortadas/` y generaremos un
embedding facial para cada cara usando un modelo de reconocimiento facial preentrenado.
Esos embeddings s alimentaran el VP-tree del Notebook 4.